In [1]:
%pip install pandas scikit-learn xgboost

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
print("0/4 complete")
# 1. Load, Merge, and FUSE CYBER + PHYSICAL
results_dir = Path(r"d:\Projects\Federated Learning\rpi\experiments\results")
csv_files = list(results_dir.rglob("cyber_data.csv"))

dfs = []
for file in csv_files:
    try:
        # Load Cyber
        cyber_df = pd.read_csv(file)
        cyber_df['window_start_time'] = pd.to_datetime(cyber_df['window_start_time'])
        cyber_df = cyber_df.sort_values('window_start_time')
        
        # Temporal Cyber Features
        cyber_df['packet_rate_diff'] = cyber_df['packet_rate'].diff().fillna(0)
        cyber_df['seq_gap_diff'] = cyber_df['sequence_number_gap'].diff().fillna(0)
        cyber_df['crc_failure_diff'] = cyber_df['crc_failure_count'].diff().fillna(0)
        cyber_df['mean_interarrival_diff'] = cyber_df['mean_interarrival_time'].diff().fillna(0)

        # Load Physical
        physical_path = str(file).replace('cyber_data.csv', 'physical_data.csv')
        phys_df = pd.read_csv(physical_path)
        phys_df['Timestamp'] = pd.to_datetime(phys_df['Timestamp'])
        phys_df = phys_df.sort_values('Timestamp')
        
        # Drop redundant/clashing columns from physical
        cols_to_drop = [c for c in ['Label', 'Node ID'] if c in phys_df.columns]
        phys_df = phys_df.drop(columns=cols_to_drop)

        # SENSOR FUSION: Snap the closest physical reading to each cyber window!
        merged_df = pd.merge_asof(cyber_df, phys_df, left_on='window_start_time', right_on='Timestamp', direction='nearest')
        
        dfs.append(merged_df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

master_df = pd.concat(dfs, ignore_index=True)

# 2. Balance Dataset (Random Undersampling)
min_class_size = master_df['AttackLabel'].value_counts().min()
balanced_df = master_df.groupby('AttackLabel').sample(n=min_class_size, random_state=42)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("1/4 complete")
# 3. Clean Data (Drop Identifiers)
columns_to_drop = [
    'window_id', 'window_start_time', 'window_end_time', 'Timestamp', 
    'node_id', 'src_ip', 'dst_ip', 'src_port', 'dst_port'
]
existing_cols = [col for col in columns_to_drop if col in balanced_df.columns]
clean_df = balanced_df.drop(columns=existing_cols)

# Drop any rows where physical data was totally missing (NaNs from merge)
clean_df = clean_df.dropna()

# 4. Split Features and Labels
X = clean_df.drop(columns=['AttackLabel'])
y = clean_df['AttackLabel']
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
target_names = label_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print("2/4 complete")
# 5. Scale Data (Crucial for SVM, KNN, MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train Fused Models
print(f"Training on {len(X_train)} fully fused Cyber-Physical samples!\n")

print("Training XGBoost (The Cyber Champion)...")
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)

print("Training Neural Network (MLP)...")
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp_model.fit(X_train_scaled, y_train)
mlp_preds = mlp_model.predict(X_test_scaled)
mlp_acc = accuracy_score(y_test, mlp_preds)
print("3/4 complete")
# 7. Compare Fused Models
print("\n" + "="*50)
print(f"🏆 FUSED CYBER-PHYSICAL MODEL RESULTS 🏆")
print("="*50)
print(f"XGBoost Accuracy      : {xgb_acc * 100:.2f}%")
print(f"Neural Net Accuracy   : {mlp_acc * 100:.2f}%")
print("="*50)

print("\n--- XGBoost Classification Report ---")
print(classification_report(y_test, xgb_preds, target_names=target_names))


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


0/4 complete
1/4 complete
2/4 complete
Training on 13780 fully fused Cyber-Physical samples!

Training XGBoost (The Cyber Champion)...
Training Neural Network (MLP)...
3/4 complete

🏆 FUSED CYBER-PHYSICAL MODEL RESULTS 🏆
XGBoost Accuracy      : 99.85%
Neural Net Accuracy   : 99.45%

--- XGBoost Classification Report ---
                           precision    recall  f1-score   support

ConnectionResetExperiment       0.99      1.00      1.00       313
  DataTamperingExperiment       0.99      1.00      1.00       313
          DelayExperiment       1.00      1.00      1.00       313
    DeviceSpoofExperiment       1.00      1.00      1.00       314
DuplicatePacketExperiment       1.00      1.00      1.00       313
       FloodingExperiment       1.00      1.00      1.00       313
                   Normal       1.00      0.99      1.00       314
PacketInjectionExperiment       1.00      1.00      1.00       314
     PacketLossExperiment       1.00      1.00      1.00       313
       